In [0]:
from pyspark.sql.functions import coalesce, try_to_date, col, trim, upper

In [0]:
df_exchange = spark.table("retailer.bronze.exchange_rates_raw")

display(df_exchange)

In [0]:
df_exchange_clean = df_exchange.withColumn(
    "date",
    coalesce(
        try_to_date(col("date"), "dd-MM-yyyy"),
        try_to_date(col("date"), "M/d/yyyy")
    )
)

In [0]:
df_exchange_clean = df_exchange_clean.withColumn(
    "currency",
    trim(upper(col("currency")))
)

In [0]:
df_exchange_clean = (
    df_exchange_clean
    .filter(col("date").isNotNull() & col("currency").isNotNull())
    .dropDuplicates(["date", "currency"])
)

In [0]:
display(df_exchange_clean)

df_exchange_clean.printSchema()

In [0]:
df_exchange_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retailer.silver.exchange_rates")
